<a href="https://colab.research.google.com/github/ajaykumar080286/feature_engineering/blob/main/15_titanic_using_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [194]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV


In [40]:
df=pd.read_csv("https://raw.githubusercontent.com/ajaykumar080286/feature_engineering/main/train.csv")

In [41]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [42]:
df.drop(columns=["PassengerId","Name","Cabin","Ticket"], inplace=True)

In [43]:
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


In [44]:
X=df.iloc[:,1:8]
y=df.iloc[:,0]

In [45]:
X_train,X_test, y_train, y_test= train_test_split(X,y, test_size=0.2, random_state=42)

In [46]:
X_train.head(1)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S


In [147]:
trf1=ColumnTransformer([
    ("simpleImp_age",SimpleImputer(),[2]),
    ("simpleImp_embarked",SimpleImputer(strategy="most_frequent"),[6])
], remainder="passthrough")

In [170]:
# one hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,3])
],remainder='passthrough')

In [171]:
trf3=ColumnTransformer([
    ("scale",MinMaxScaler(),slice(0,10))
], remainder="passthrough")

In [172]:
trf4 = SelectKBest(score_func=chi2,k=8)

In [173]:
trf5 = DecisionTreeClassifier()

**Create Pipeline**

In [174]:
pipe=Pipeline([
    ("trf1", trf1),
     ("trf2", trf2),
     ("trf3", trf3),
     ("trf4", trf4),
     ("trf5", trf5)
])

In [175]:
#pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [178]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('simpleImp_age',
                                                  SimpleImputer(), [2]),
                                                 ('simpleImp_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 3])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7d55eb3ef2e0>)),
                ('trf5', DecisionTreeClassifier())])

In [177]:
from sklearn import set_config
set_config(display='diagram')

In [179]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('simpleImp_age', SimpleImputer(), [2]),
                                 ('simpleImp_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 3])]),
 'trf3': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x7d55eb3ef2e0>),
 'trf5': DecisionTreeClassifier()}

In [180]:
y_pred=pipe.predict(X_test)

In [181]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7877094972067039

In [190]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_


array([29.49884615])

In [191]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_


array(['S'], dtype=object)

In [192]:
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.7852752880921896)

***GridSearch using Pipeline***

In [193]:
params = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [195]:
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')

In [196]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('simpleImp_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('simpleImp_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          3])])),
                                       ('trf3',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x7d55eb3ef2e0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [197]:
grid.best_score_

np.float64(0.8033093666896484)

In [198]:
grid.best_params_

{'trf5__max_depth': 3}

**Exporting the Pipeline**

In [199]:
# export
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))

In [200]:
pipe = pickle.load(open('pipe.pkl','rb'))

In [201]:
test_input2 = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'],dtype=object).reshape(1,7)

In [202]:
pipe.predict(test_input2)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])